# 14장. 반복되는 분석 흐름을 자동화하기

이 노트북은 `book/chapters/ch14_airflow_pipeline.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 반복되는 분석 작업을 **입력 확인 → 전처리 → 분석 → 시각화 → 보고서 생성 → 산출물 검증** 단계로 나누고, Python 스크립트와 Airflow DAG로 실행 순서를 관리하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 장은 먼저 Airflow 없이 Python 스크립트만으로 전체 흐름을 검증합니다.
- 그다음 같은 흐름을 Airflow DAG로 연결하는 구조를 확인합니다.
- Docker는 사용하지 않고, 로컬 Python 가상환경과 `airflow standalone` 기준으로 설명합니다.
- Windows에서는 PowerShell 네이티브 실행보다 WSL2 Ubuntu 환경을 권장합니다.
- Airflow 설치 전에도 이 노트북과 `python scripts/run_ch14_pipeline.py`는 실행할 수 있습니다.


## 1. 자동화는 코드를 대신 쓰는 일이 아니다

자동화는 분석 코드를 없애는 것이 아니라, 잘 정리된 분석 코드를 정해진 순서로 실행하도록 만드는 일입니다. 전처리, 분석, 시각화, 보고서 생성 코드가 뒤섞여 있으면 자동화하기 어렵습니다.

| 단계 | 하는 일 | 입력 | 출력 |
|---|---|---|---|
| 입력 확인 | 원본 데이터가 있는지 확인 | `data/raw/*.csv` | 확인 결과 |
| 전처리 | 결측치, 타입, 중복 처리 | 원본 CSV | `data/processed/*_clean.csv` |
| 분석 | 주요 지표 계산 | 전처리 데이터 | `reports/*.csv` |
| 시각화 | 그래프 생성 | 분석 결과 CSV | `reports/figures/*.png` |
| 보고서 | Markdown 보고서 작성 | 표, 그래프, 해석 문장 | `reports/*.md` |
| 검증 | 결과 파일 존재 여부 확인 | 산출물 목록 | 검증 로그 |
| 전달 | 메일, Slack, Drive 등으로 공유 | 보고서 파일 | 알림 또는 발송 기록 |


## 2. Make, n8n, Airflow의 역할 구분

자동화 도구는 비슷해 보이지만 잘 맞는 상황이 다릅니다.

| 도구 | 잘 맞는 상황 | 예시 |
|---|---|---|
| Make | 외부 서비스 연결과 알림 자동화 | 보고서 파일 생성 후 Gmail 발송, Slack 알림 |
| n8n | 노코드/로우코드 기반 워크플로우 구성 | API 호출, 데이터 저장, 내부 도구 연결 |
| Airflow | 코드 기반 데이터 파이프라인 운영 | 전처리 → 분석 → 시각화 → 보고서 생성 순서 관리 |

실무에서는 Airflow가 분석 파이프라인을 실행하고, Make나 n8n이 결과 보고서를 외부 서비스로 전달하는 식으로 나눠 사용할 수 있습니다.


## 3. 이번 장에서 완성할 파이프라인

이번 실습에서는 온라인 쇼핑몰 분석 흐름을 다음 순서로 자동화합니다.

```text
check_input_files
→ run_preprocessing
→ run_analysis
→ generate_visualizations
→ generate_report
→ validate_outputs
```

Airflow를 사용하기 전, 먼저 Python 함수와 스크립트로 이 순서가 정상 실행되는지 확인합니다.


## 4. 패키지와 경로 설정

프로젝트 루트, 원본 데이터 폴더, 전처리 데이터 폴더, 보고서 폴더, Airflow DAG 폴더를 설정합니다.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
FIGURE_DIR = REPORT_DIR / 'figures'
AIRFLOW_DIR = PROJECT_ROOT / '.airflow'
DAG_DIR = AIRFLOW_DIR / 'dags'

for path in [RAW_DIR, PROCESSED_DIR, REPORT_DIR, FIGURE_DIR, DAG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)
print('Airflow DAG 폴더:', DAG_DIR)


## 5. 파이프라인 Task 구조 확인하기

`src/automation_pipeline.py`에는 이번 장에서 사용할 공통 함수가 들어 있습니다. 먼저 Task 구조를 표로 확인합니다.


In [ ]:
from src.automation_pipeline import (
    check_input_files,
    create_airflow_setup_guide,
    create_pipeline_task_summary,
    generate_report,
    generate_visualizations,
    get_project_paths,
    run_analysis,
    run_local_pipeline,
    run_preprocessing,
    validate_outputs,
)

task_summary = create_pipeline_task_summary()
task_summary.to_csv(REPORT_DIR / 'ch14_pipeline_task_summary.csv', index=False, encoding='utf-8-sig')
task_summary


## 6. 입력 파일 확인

Airflow 파이프라인의 첫 단계는 원본 CSV 4개가 있는지 확인하는 것입니다. 입력 파일이 없으면 뒤의 Task를 실행하지 않아야 합니다.

필요한 파일:

```text
data/raw/customers.csv
data/raw/products.csv
data/raw/orders.csv
data/raw/order_items.csv
```

파일이 없다면 먼저 `python scripts/generate_sample_data.py`를 실행하세요.


In [ ]:
input_check = check_input_files(PROJECT_ROOT)
input_check


## 7. 전처리 Task 실행

원본 데이터를 읽고 문자열 공백, 날짜, 숫자형, `line_total`을 정리한 뒤 `data/processed/`에 저장합니다.


In [ ]:
preprocessing_outputs = run_preprocessing(PROJECT_ROOT)
preprocessing_outputs


## 8. 분석 Task 실행

전처리된 데이터를 읽어 일자별 매출과 카테고리별 매출을 계산합니다.


In [ ]:
analysis_outputs = run_analysis(PROJECT_ROOT)
analysis_outputs


In [ ]:
daily_sales = pd.read_csv(REPORT_DIR / 'ch14_daily_sales.csv')
category_sales = pd.read_csv(REPORT_DIR / 'ch14_category_sales.csv')

display(daily_sales.head())
display(category_sales.head())


## 9. 시각화 Task 실행

일자별 매출 추이를 그래프로 저장합니다. 운영체제나 폰트 설정에 따른 한글 깨짐을 줄이기 위해 그래프 제목과 축 이름은 영어로 저장합니다.


In [ ]:
figure_outputs = generate_visualizations(PROJECT_ROOT)
figure_outputs


## 10. 보고서 생성 Task 실행

일자별 매출, 카테고리별 매출, Task 요약을 바탕으로 Markdown 보고서를 생성합니다. 자동 보고서는 결과 전달 속도를 높이지만, 해석의 최종 책임은 사람에게 있습니다.


In [ ]:
report_path = generate_report(PROJECT_ROOT)
report_path


## 11. 산출물 검증 Task 실행

자동화에서 중요한 것은 실행 성공만이 아니라 필요한 산출물이 실제로 생성되었는지 확인하는 것입니다. 파일 존재 여부와 크기를 검증합니다.


In [ ]:
validation_log = validate_outputs(PROJECT_ROOT)
validation_log


## 12. 전체 파이프라인을 한 번에 실행하기

위에서 단계별로 실행한 작업은 `run_local_pipeline()` 함수로 한 번에 실행할 수 있습니다. Airflow에 연결하기 전에 이 함수가 정상 실행되어야 합니다.


In [ ]:
pipeline_result = run_local_pipeline(PROJECT_ROOT)
pipeline_result['validation_log']


터미널에서는 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_ch14_pipeline.py
```

이 단계에서 오류가 난다면 Airflow 문제가 아니라 Python 스크립트 문제입니다. 먼저 여기서 해결해야 합니다.


## 13. Airflow DAG 템플릿 확인

이번 장에서는 DAG 템플릿을 `dags/ch14_local_analysis_pipeline.py`에 추가했습니다. 로컬 Airflow에서 사용하려면 다음 중 하나를 선택합니다.

1. `dags/ch14_local_analysis_pipeline.py`를 `.airflow/dags/ch14_local_analysis_pipeline.py`로 복사
2. Airflow의 DAG 폴더를 프로젝트의 `dags/` 폴더로 설정

`.airflow/` 폴더는 로그, DB, 설정 파일이 생기므로 일반적으로 Git에 올리지 않습니다.


In [ ]:
dag_template_path = PROJECT_ROOT / 'dags' / 'ch14_local_analysis_pipeline.py'
print('DAG 템플릿 존재:', dag_template_path.exists())
print('DAG 템플릿 경로:', dag_template_path)


## 14. 로컬 Airflow 설치 흐름

Airflow는 의존성이 많은 애플리케이션이므로 기존 `.venv`와 분리해 별도의 가상환경을 만드는 것을 권장합니다. 아래 명령은 macOS, Linux, WSL2 Ubuntu 기준입니다.

```bash
python3 -m venv .venv-airflow
source .venv-airflow/bin/activate
python -m pip install --upgrade pip
```

Airflow는 설치 시 Python 버전과 Airflow 버전에 맞는 constraints 파일을 사용하는 방식이 안정적입니다. 실습 시점에는 Airflow 공식 문서에서 최신 안정 버전과 Python 지원 버전을 확인한 뒤 버전 번호를 조정하세요.

```bash
AIRFLOW_VERSION=3.3.0
PYTHON_VERSION="$(python -c 'import sys; print(f"{sys.version_info.major}.{sys.version_info.minor}")')"
CONSTRAINT_URL="https://raw.githubusercontent.com/apache/airflow/constraints-${AIRFLOW_VERSION}/constraints-${PYTHON_VERSION}.txt"

pip install "apache-airflow==${AIRFLOW_VERSION}" --constraint "${CONSTRAINT_URL}"
pip install pandas matplotlib
```


## 15. Airflow 실행 순서

프로젝트 루트에서 아래 순서로 실행합니다.

```bash
source .venv-airflow/bin/activate
export AIRFLOW_HOME="$(pwd)/.airflow"
mkdir -p "$AIRFLOW_HOME/dags"
cp dags/ch14_local_analysis_pipeline.py .airflow/dags/ch14_local_analysis_pipeline.py
airflow standalone
```

Airflow 웹 UI는 보통 아래 주소에서 확인합니다.

```text
http://localhost:8080
```

새 터미널에서 DAG가 인식되는지 확인합니다.

```bash
source .venv-airflow/bin/activate
export AIRFLOW_HOME="$(pwd)/.airflow"
airflow dags list | grep ch14
```


## 16. Airflow UI 또는 CLI에서 DAG 실행

웹 UI에서 실행하는 방법:

1. `http://localhost:8080` 접속
2. `ch14_local_analysis_pipeline` DAG 찾기
3. DAG 활성화
4. 수동 실행 버튼 클릭
5. Graph 또는 Grid 화면에서 Task 실행 순서 확인
6. 실패한 Task가 있다면 로그 확인

CLI로 실행하려면 다음 명령을 사용합니다.

```bash
airflow dags trigger ch14_local_analysis_pipeline
```

결과 검증 로그는 아래 파일에서 확인합니다.

```bash
cat reports/ch14_airflow_validation_log.csv
```


## 17. 실패 상황을 일부러 만들어 보기

자동화 실습에서 중요한 것은 성공보다 실패를 읽는 능력입니다. 다음처럼 입력 파일 하나를 잠시 다른 이름으로 바꿔 봅니다.

```bash
mv data/raw/customers.csv data/raw/customers_backup.csv
airflow dags trigger ch14_local_analysis_pipeline
```

이번에는 `check_input_files` 단계에서 실패해야 합니다. Airflow UI에서 실패한 Task를 클릭하고 로그를 확인합니다. 실습이 끝나면 파일명을 다시 복구합니다.

```bash
mv data/raw/customers_backup.csv data/raw/customers.csv
```


## 18. Make와 n8n은 전달과 연결에 강하다

Airflow가 코드 기반 분석 파이프라인을 담당한다면, Make와 n8n은 결과물을 외부 서비스와 연결하는 데 유용합니다.

| 구간 | 담당 도구 예시 | 역할 |
|---|---|---|
| 데이터 처리 | Python, Airflow | 전처리, 분석, 시각화, 보고서 생성 |
| 결과 검증 | Python, Airflow | 파일 생성 여부, 크기, 로그 확인 |
| 외부 전달 | Make, n8n | 메일 발송, Slack 알림, Drive 업로드 |
| 운영 확인 | Airflow UI, Make/n8n 실행 로그 | 실패 지점과 재실행 여부 확인 |

Make나 n8n에서 모든 분석을 처리하려고 하면 복잡해질 수 있습니다. 반대로 Airflow에서 외부 앱 연계까지 모두 처리하려고 해도 운영이 무거워질 수 있습니다. 분석 처리와 외부 전달을 나누면 구조가 단순해집니다.


## 19. LLM에게 파이프라인 설계를 요청하는 프롬프트

LLM은 자동화 파이프라인 설계 초안을 만드는 데 도움을 줄 수 있습니다. 단, 파일 경로, 실행 환경, 실제 컬럼명, API 인증, 발송 권한은 사람이 확인해야 합니다.

```text
온라인 쇼핑몰 주문 데이터를 매일 분석하는 자동화 파이프라인을 설계하려고 합니다.

입력 파일:
- orders.csv
- order_items.csv
- products.csv
- customers.csv

필요한 작업:
- 입력 파일 확인
- 데이터 전처리
- 매출 분석
- 시각화 생성
- Markdown 보고서 생성
- 결과 파일 검증
- 보고서 발송 또는 Slack 알림

요청:
1. 전체 작업을 단계별 Task로 나누어 주세요.
2. 각 Task의 입력과 출력을 표로 정리해 주세요.
3. Airflow가 담당할 부분과 Make/n8n이 담당할 부분을 나누어 주세요.
4. 실패했을 때 확인해야 할 로그와 검증 항목을 제안해 주세요.
5. 지나치게 복잡한 설치 절차보다 운영 흐름 중심으로 설명해 주세요.
```


## 20. 자동화 결과를 해석하는 방법

파이프라인이 성공했다고 해서 분석 결과가 항상 타당한 것은 아닙니다. 자동화 결과를 볼 때는 다음 세 가지를 나누어 확인합니다.

| 구분 | 확인할 질문 |
|---|---|
| 실행 성공 | 모든 Task가 성공했는가? |
| 산출물 성공 | 필요한 CSV, 그래프, 보고서가 생성되었는가? |
| 분석 품질 | 결과 수치와 해석이 데이터에 맞는가? |

Airflow UI에서 모든 Task가 초록색이어도 보고서의 해석이 잘못되었거나 CSV 값이 비어 있으면 분석 품질은 낮습니다.


## 21. 실습 과제

아래 과제를 직접 해결해 보세요.

1. `python scripts/run_ch14_pipeline.py`를 실행하고 생성된 산출물을 확인하세요.
2. `reports/ch14_airflow_validation_log.csv`에서 모든 `status`가 `ok`인지 확인하세요.
3. `dags/ch14_local_analysis_pipeline.py`를 `.airflow/dags/`로 복사하고 Airflow에서 DAG가 보이는지 확인하세요.
4. 입력 파일 하나를 임시로 변경해 `check_input_files` 실패를 확인하세요.
5. `ch14_airflow_report.md`에 원인 단정 문장이 없는지 검토하세요.
6. Make 또는 n8n으로 보고서 파일을 Slack이나 Gmail로 전달한다면 어떤 단계가 필요한지 표로 정리하세요.


In [ ]:
# 과제 1. 생성된 산출물 목록을 확인해 보세요.
for path in sorted(REPORT_DIR.glob('ch14_*')):
    print(path.name, path.stat().st_size if path.exists() else 'missing')


## 22. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 반복 분석 업무를 Task로 나누는 방법
- Make, n8n, Airflow의 역할 구분
- 입력 확인, 전처리, 분석, 시각화, 보고서, 검증 단계 구성
- Airflow 없이 Python 스크립트로 먼저 전체 흐름 검증
- `src/automation_pipeline.py` 공통 모듈 구성
- Airflow가 호출할 `scripts/ch14_*.py` 단계별 스크립트 구성
- `dags/ch14_local_analysis_pipeline.py` DAG 템플릿 구성
- `airflow standalone` 기반 로컬 실습 흐름
- 실패 Task 로그 확인과 산출물 검증
- 자동화 결과의 실행 성공, 산출물 성공, 분석 품질 구분

다음 장에서는 지금까지 배운 EDA, 시각화, 머신러닝, LLM 활용, 외부 데이터, 자동화 아이디어를 기말 프로젝트로 통합합니다.
